## RF optimisation

In [1]:
import geemap
import ee

#ee.Authenticate(force=True)
ee.Initialize(project='ee-sbouchiba')

In [2]:
## Import Geneva footprint
import geopandas as gpd
try:
  import json
except:
  !pip install json
  import json

geneva_roi_gpd = gpd.read_file('./input/geneva_footprint.geojson')
#print(geneva_roi_gpd)
#print(geneva_roi_gpd.crs)
geneva_roi_gpd = geneva_roi_gpd.to_crs("EPSG:4326") # convert projected CRS into WGS84 longitude/latitude.


geom = geneva_roi_gpd.geometry.iloc[0]
roi = ee.Geometry(geom.__geo_interface__)

roi_fc = ee.FeatureCollection([ee.Feature(roi)])


In [3]:
## Import train/test datasets

#train_polygons = ee.FeatureCollection('./input/geneva_2022_training_polygons.geojson');
#test_polygons  = ee.FeatureCollection('./input/geneva_2022_testing_polygons.geojson');
import geopandas as gpd


train_centroids_gdf = gpd.read_file('./input/geneva_2022_training_centroids.geojson')
train_polygons_gdf = gpd.read_file('./input/geneva_2022_training_polygons.geojson')
test_centroids_gdf = gpd.read_file('./input/geneva_2022_testing_centroids.geojson')
test_polygons_gdf = gpd.read_file('./input/geneva_2022_testing_polygons.geojson')

#print(train_centroids_gdf.columns)

# Remove signature class for test datasets
train_centroids_gdf = (train_centroids_gdf[train_centroids_gdf["macroclass_id"] != 1].copy())
train_polygons_gdf = (train_polygons_gdf[train_polygons_gdf["macroclass_id"] != 1].copy())

# Re-map: define new class mapping that is simpler to handle
class_mapping = {
    2: 1,   # Compact MR
    5: 2,   # Open MR
    6: 3,   # Open LR
    8: 4,   # Large LR
    11: 5,  # Dense Trees
    14: 6,  # Low plants
    15: 7,  # Bare Rock/Paved
    17: 8   # Water
}
train_centroids_gdf["LCZ_class"] = (train_centroids_gdf["macroclass_id"].map(class_mapping))
train_polygons_gdf["LCZ_class"]  = (train_polygons_gdf["macroclass_id"].map(class_mapping))
test_centroids_gdf["LCZ_class"]  = (test_centroids_gdf["macroclass"].map(class_mapping))
test_polygons_gdf["LCZ_class"]   = (test_polygons_gdf["macroclass"].map(class_mapping))
# Some monitoring
#print(train_centroids_gdf.columns.tolist())
#print(train_centroids_gdf["LCZ_class"].value_counts().sort_index())
#print(test_centroids_gdf["LCZ_class"].value_counts().sort_index())
#print(train_centroids_gdf.shape)
#print(test_centroids_gdf.shape)
# Convert to EE datasets
train_centroids = geemap.geopandas_to_ee(train_centroids_gdf)
train_polygons  = geemap.geopandas_to_ee(train_polygons_gdf)
test_centroids  = geemap.geopandas_to_ee(test_centroids_gdf)
test_polygons   = geemap.geopandas_to_ee(test_polygons_gdf)

#type(train_centroids)


In [4]:
year = 2022;
startDate = ee.Date.fromYMD(year, 1, 1);
endDate = startDate.advance(1, 'year');

embeddings = ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL');
embeddingsFiltered = embeddings\
  .filter(ee.Filter.date(startDate, endDate))\
  .filter(ee.Filter.bounds(roi));

embeddingsImage = embeddingsFiltered.mosaic();
#embeddingsImage = embeddings.mosaic();


#Overlay the samples on the image to get training data.

#

In [50]:
import numpy as np
import pandas as pd
# Metrics to collect

#numberOfTrees_list=[100,300,500];
#variablesPerSplit_list=[4, 8, 16, 32];
#minLeafPopulation_list=[1, 5, 10]

numberOfTrees_list=[10,20];
variablesPerSplit_list=[8];
minLeafPopulation_list=[5]

#exportFolder = './assets/bouchiba/rf/';

results = []

for i in range(len(numberOfTrees_list)):
    for j in range(len(variablesPerSplit_list)):
        for k in range(len(minLeafPopulation_list)):
            
            numberOfTrees     = numberOfTrees_list[i]
            variablesPerSplit = variablesPerSplit_list[j]
            minLeafPopulation = minLeafPopulation_list[k]
            setup = f"({numberOfTrees},{variablesPerSplit},{minLeafPopulation})"
            print(f"--------")
            print(f"Processing classifier setup: {setup}")
            classifier = ee.Classifier.smileRandomForest(numberOfTrees=numberOfTrees,
                                                         variablesPerSplit = variablesPerSplit,
                                                         minLeafPopulation = minLeafPopulation).\
                                        train(features=training, 
                                        classProperty='LCZ_class', 
                                        inputProperties=embeddingsImage.bandNames())
            print(f"Compute and save classified image..")
            # Compute and save classified image
            classified = embeddingsImage.classify(classifier)

            # Validation set metrics
            print(f"Computation validation metrics..")
            validation_set = classified.sampleRegions(collection=test_centroids,
                                                      properties=['LCZ_class'],
                                                      scale=10)
            confusion_matrix = validation_set.errorMatrix('LCZ_class','classification')
            # EE built in producersAccuracy().getInfo() and consumersAccuracy().getInfo()
            # are in an annoying nested list form -> I redefine myself. I checked they are
            # coherent and giving the same results.
            
            cm = np.array(confusion_matrix.getInfo())
            # EE internally allocates first row and first column of zeroes for some reason.-> remove them
            cm = cm[1:, 1:]
            diag = np.diag(cm)
            PA = 100 * diag / cm.sum(axis=1)
            UA = 100 * diag / cm.sum(axis=0)
            OA = 100 * diag.sum() / cm.sum()
            # Training set (check for overfitting)
            training_set = classified.sampleRegions(collection=train_centroids,
                                                    properties=['LCZ_class'],
                                                    scale=10)

            confusion_matrix_train = training_set.errorMatrix('LCZ_class','classification')
            cmt = np.array(confusion_matrix_train.getInfo())
            # EE internally allocates first row and first column of zeroes for some reason.-> remove them
            cmt = cmt[1:, 1:]
            diag = np.diag(cmt)
            PAt = 100 * diag / cmt.sum(axis=1)
            UAt = 100 * diag / cmt.sum(axis=0)
            OAt = 100 * diag.sum() / cmt.sum()
            
            results.append({
                "numberOfTrees": numberOfTrees,
                "variablesPerSplit": variablesPerSplit,
                "minLeafPopulation": minLeafPopulation,
                "OA_train": OAt,
                "UA_train": UAt,
                "PA_train": PAt,
                "OA": OA,
                "UA": UA,
                "PA": PA
            })
            print(f"Metrics computed and appened to results dataframe!")
print("-------")
print("Done!")

# Load asset of classified image:

            

df = pd.DataFrame(results)
       

--------
Processing classifier setup: (10,8,5)
Compute and save classified image..
Computation validation metrics..
Metrics computed and appened to results dataframe!
--------
Processing classifier setup: (20,8,5)
Compute and save classified image..
Computation validation metrics..
Metrics computed and appened to results dataframe!
Done!


In [29]:
df.to_csv("./tables/RF_opti_Genveva2022_1.csv")

In [51]:
df.to_pickle("./tables/RF_opti_Geneva2022_mini.pkl")

In [36]:
df

,numberOfTrees,variablesPerSplit,minLeafPopulation,OA_train,UA_train,PA_train,OA,UA,PA
0,100,4,1,0.997333,"[[0], [1], [0.9791666666666666], [1], [1], [1]...","[[0, 0.9791666666666666, 1, 1, 1, 1, 1, 1, 1]]",0.896774,"[[0], [1], [0.7894736842105263], [0.8947368421...","[[0, 0.8636363636363636, 0.7894736842105263, 0..."
1,100,4,10,0.938667,"[[0], [0.9787234042553191], [0.916666666666666...","[[0, 0.8846153846153846, 0.88, 0.9333333333333...",0.845161,"[[0], [1], [0.7368421052631579], [0.7894736842...","[[0, 0.7037037037037037, 0.6666666666666666, 0..."
2,100,8,1,0.997333,"[[0], [1], [0.9791666666666666], [1], [1], [1]...","[[0, 0.9791666666666666, 1, 1, 1, 1, 1, 1, 1]]",0.890323,"[[0], [1], [0.7368421052631579], [0.8947368421...","[[0, 0.8636363636363636, 0.875, 0.809523809523..."
3,100,8,10,0.930667,"[[0], [0.9787234042553191], [0.875], [0.914893...","[[0, 0.8679245283018868, 0.875, 0.895833333333...",0.864516,"[[0], [1], [0.7368421052631579], [0.8421052631...","[[0, 0.7307692307692307, 0.8235294117647058, 0..."
4,100,32,1,0.997333,"[[0], [1], [0.9791666666666666], [1], [1], [1]...","[[0, 0.9791666666666666, 1, 1, 1, 1, 1, 1, 1]]",0.870968,"[[0], [1], [0.7894736842105263], [0.8421052631...","[[0, 0.8260869565217391, 0.8333333333333334, 0..."
5,100,32,10,0.922667,"[[0], [0.9148936170212766], [0.875], [0.914893...","[[0, 0.86, 0.8571428571428571, 0.9148936170212...",0.877419,"[[0], [1], [0.7894736842105263], [0.8421052631...","[[0, 0.76, 0.8333333333333334, 0.8421052631578..."
6,300,4,1,0.997333,"[[0], [1], [0.9791666666666666], [1], [1], [1]...","[[0, 0.9791666666666666, 1, 1, 1, 1, 1, 1, 1]]",0.903226,"[[0], [1], [0.7894736842105263], [0.8947368421...","[[0, 0.8260869565217391, 0.8823529411764706, 0..."
7,300,4,10,0.941333,"[[0], [0.9787234042553191], [0.9375], [0.91489...","[[0, 0.8679245283018868, 0.9, 0.93478260869565...",0.851613,"[[0], [1], [0.7894736842105263], [0.7894736842...","[[0, 0.7307692307692307, 0.7142857142857143, 0..."
8,300,8,1,0.997333,"[[0], [1], [0.9791666666666666], [1], [1], [1]...","[[0, 0.9791666666666666, 1, 1, 1, 1, 1, 1, 1]]",0.903226,"[[0], [1], [0.7894736842105263], [0.8947368421...","[[0, 0.8636363636363636, 0.8823529411764706, 0..."
9,300,8,10,0.941333,"[[0], [0.9787234042553191], [0.916666666666666...","[[0, 0.9019607843137255, 0.9166666666666666, 0...",0.877419,"[[0], [1], [0.7894736842105263], [0.8947368421...","[[0, 0.76, 0.8823529411764706, 0.8095238095238..."
